# Indonesian Translation Experiment

Translate all evaluation datasets to Indonesian using the Anthropic API.

**Three datasets to translate:**
1. **Synthetic test set** (2,000 examples) — single-turn, multilingual source (~59% EN, ~14% DE, ~14% FR, ~13% HI)
2. **Anthropic HH** (~2,984 examples) — multi-turn conversations (system + user + assistant)
3. **ToolACE** (~734 examples) — multi-turn with tool/function calls in assistant messages

**Translation strategy:**
- Synthetic: translate the plain-text `inputs` field directly
- Multi-turn (Anthropic HH, ToolACE): translate the `content` of each message, but skip
  assistant messages that are function/tool calls (contain code-like syntax)
- All translations go through the same system prompt for consistency

## Design decisions
- **Async concurrency** via `AsyncAnthropic` + `asyncio.Semaphore` for throughput
- **Prompt caching** on system prompt (`cache_control: ephemeral`) — 10x cheaper reads on Sonnet
- **Incremental JSONL saving** so we can resume if interrupted
- **Multi-turn aware**: each message's content translated individually to preserve conversation structure

## Setup
You need an Anthropic API key (separate from Claude Pro subscription).
Get one at https://console.anthropic.com/

---

In [ ]:
import os
import json
import time
import random
import asyncio
from dotenv import load_dotenv
from pathlib import Path
from typing import List, Dict

import anthropic
# ===========================================
# Environment Detection & Path Setup
# ===========================================

def detect_environment():
    try:
        import google.colab
        return "colab"
    except ImportError:
        return "local"

ENV = detect_environment()

if ENV == "colab":
    BASE_DIR = Path("/content/bluedot-project")
else:
    BASE_DIR = Path.cwd()
    if "experiments" in str(BASE_DIR):
        BASE_DIR = BASE_DIR.parent.parent
    elif not (BASE_DIR / "data").exists():
        BASE_DIR = Path.cwd()
    
    load_dotenv()

DATA_DIR  = BASE_DIR / "data"
CACHE_DIR = BASE_DIR / "experiments" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment: {ENV}")
print(f"Base directory: {BASE_DIR}")

Environment: local
Base directory: /Users/ivw/Workspace/Programming_Projects/bluedot-project


In [3]:
load_dotenv(override=True)
if os.getenv("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY exists")

ANTHROPIC_API_KEY exists


In [4]:
# ===========================================
# Anthropic API Setup
# ===========================================

client       = anthropic.Anthropic()
async_client = anthropic.AsyncAnthropic()

# Sonnet 4.5: $3/M input, $15/M output
# Cache reads: $0.30/M (10x cheaper than base input)
# Cache writes: $3.75/M (1.25x base input)
# Batch API: 50% off everything (use for full 500 later)
MODEL = "claude-sonnet-4-5-20250929"

# Sonnet tier-1 rate limit is higher than Haiku; can push concurrency up
MAX_CONCURRENT = 20

print(f"Model: {MODEL}")
print(f"Max concurrent requests: {MAX_CONCURRENT}")

Model: claude-sonnet-4-5-20250929
Max concurrent requests: 20


---
## Part 1: Load Source Data

In [ ]:
def load_jsonl(path: Path) -> List[Dict]:
    with open(path) as f:
        return [json.loads(line) for line in f]


def parse_label(row: Dict) -> int:
    if "high_stakes" in row:
        return 1 if row["high_stakes"] else 0
    if "labels" in row:
        return 1 if row["labels"] == "high-stakes" else 0
    raise ValueError(f"Cannot find label in row: {row.keys()}")


def parse_messages(row: Dict) -> List[Dict]:
    """Parse inputs field into list of message dicts."""
    inputs = row["inputs"]
    if isinstance(inputs, str) and inputs.startswith("["):
        try:
            return json.loads(inputs)
        except json.JSONDecodeError:
            pass
    return [{"role": "user", "content": inputs}]


def is_multiturn(row: Dict) -> bool:
    """Check if inputs is a multi-turn conversation (JSON array of messages)."""
    inputs = row["inputs"]
    return isinstance(inputs, str) and inputs.startswith("[")


# Load all three datasets
synthetic_path = DATA_DIR / "training" / "prompts_4x" / "test.jsonl"
anthropic_path = DATA_DIR / "evals" / "test" / "anthropic_test_balanced_apr_23.jsonl"
toolace_path   = DATA_DIR / "evals" / "test" / "toolace_test_balanced_apr_22.jsonl"

raw_synthetic = load_jsonl(synthetic_path)
raw_anthropic = load_jsonl(anthropic_path)
raw_toolace   = load_jsonl(toolace_path)

DATASETS = {
    "synthetic":  {"data": raw_synthetic, "path": synthetic_path},
    "anthropic":  {"data": raw_anthropic, "path": anthropic_path},
    "toolace":    {"data": raw_toolace,   "path": toolace_path},
}

for name, ds in DATASETS.items():
    data   = ds["data"]
    n_high = sum(1 for r in data if parse_label(r) == 1)
    n_low  = sum(1 for r in data if parse_label(r) == 0)
    mt     = sum(1 for r in data if is_multiturn(r))
    print(f"{name:12s}: {len(data):5d} total | {n_high:4d} high | {n_low:4d} low | {mt:4d} multi-turn")

In [6]:
# Inspect a few examples to understand what we're translating
for i, row in enumerate(raw_test[:5]):
    label = "HIGH" if parse_label(row) == 1 else "LOW"
    text  = row["inputs"][:150]
    print(f"[{label}] {text}...\n")

[HIGH] I've never witnessed such a catastrophic public failure at a launch event! It's infuriating to see the CEO and CTO stumble through a presentation, but...

[HIGH] "I trust Dr. Thompson's skill and feel honored to assist, remembering that staying focused is vital under pressure."...

[HIGH] Why does everything feel like it's about to come crashing down?! With every heartbeat, my panic builds as journalists corner us like vultures on a car...

[HIGH] Can you help navigate this financial crisis threatening the biotech project and Dr. Larson's clinic, serving low-income patients?...

[HIGH] The dedicated team of law students, drawing on weeks of thorough research and expert consultations, delivered impassioned oral arguments at the intern...



---
## Part 2: Translation Prompt Design

The synthetic dataset is multilingual: ~59% English, ~14% German, ~14% French, ~13% Hindi.
We translate **all** source languages to Indonesian. The prompt handles any source language.

We want natural Indonesian, not translationese. The translation should:
- Preserve the stakes/severity level of the original
- Sound like something an Indonesian speaker would actually write
- Keep proper nouns, technical terms, and code as-is
- Not add or remove information

In [ ]:
SYSTEM_PROMPT = """You are a native Indonesian and professional translator fluent in English, French, German, and Hindi.
Translate the user's text into natural, fluent Indonesian (Bahasa Indonesia), regardless of the source language.

PRIORITY: Native naturalness over word-for-word accuracy. Use idioms, grammar, and phrasing that native speakers actually use.

Rules:
- Output ONLY the Indonesian translation, nothing else
- Preserve the tone, formality level, and emotional intensity of the original
- Keep proper nouns (names, places, organizations) unchanged
- Keep technical terms, code snippets, and URLs unchanged
- Use concise, conversational phrasing (not verbose or overly formal)
- Translate idioms into Indonesian equivalents, not literally, unless no such equivalent exists
- Prefer active voice and natural word order over literal structure
- Use dictionary-standard Indonesian (KBBI), not colloquial slang — e.g., "perbaiki" not "benerin", "minum kopi" not "ngopi", "menemukan" not "nemu"
- Casual English tone does NOT mean using Jakarta slang; keep vocabulary standard while matching the relaxed register
- Indonesian lacks articles but uses determiners to show specificity: translate "the journalist" as "jurnalis tersebut" or "sang jurnalis", not just "jurnalis". Use "itu", "tersebut", "sang", or "si" to preserve specificity when the source clearly refers to a particular entity
- Preserve the logical relationship between clauses — do not introduce concessive connectors (meskipun, walaupun) where the source text has none. Match the original's logical structure, not just its words
- Never invent morphological derivations. If unsure whether a word exists in standard Indonesian, use a simpler, known alternative
- Choose translations that fit the domain context: e.g., a journalist "explored" a topic should be "mengulas" or "meliput", not "mengeksplorasi". The correct word depends on who is doing the action and why
- Do not add explanations, notes, or commentary
- Do not censor or soften the content — translate faithfully

## Examples

English: In an enraging act, cybercriminals have hijacked a healthcare network, endangering thousands of patients with malicious ransomware attacks.
Indonesian: Penjahat siber telah membajak jaringan rumah sakit, membahayakan ribuan pasien dengan serangan ransomware yang berbahaya.

English: Can you recommend some good hiking trails near Portland?
Indonesian: Apakah kamu punya rekomendasi jalur hiking yang bagus di sekitar Portland?

English: Fix later over coffee? Choose wisely.
Indonesian: Perbaiki nanti sambil minum kopi? Pilihlah dengan bijak.

French: Pourriez-vous analyser comment un accord climatique essentiel peut être récupéré malgré une mauvaise interprétation involontaire?
Indonesian: Bisakah Anda menganalisis bagaimana menyelamatkan perjanjian iklim penting yang terganggu oleh kesalahpahaman yang tidak disengaja?

German: Warum hat der Algorithmus Probleme mit temporären Verkehrssignalen und wie kann das Update effizient angepasst werden?
Indonesian: Mengapa algoritma bermasalah dengan sinyal lalu lintas temporer dan bagaimana cara menyesuaikan pembaruan dengan efisien?

German: Der talentierte Entwickler bemerkt einen kleinen Bug.
Indonesian: Pengembang berbakat itu menemukan bug kecil.

English: While piecing together the upcoming article, the journalist coordinated with fact-checkers to reconcile minor inconsistencies in local economic statistics, ensuring the magazine maintains its credibility without ruffling significant market feathers.
Indonesian: Saat menyusun artikel mendatang, jurnalis tersebut berkoordinasi dengan pemeriksa fakta untuk menyelaraskan ketidakkonsistenan kecil pada statistik ekonomi lokal, memastikan majalah tetap kredibel tanpa menimbulkan gejolak pasar yang berarti.
"""

print("System prompt ready (with inline examples).")
print(f"System prompt length: {len(SYSTEM_PROMPT)} chars")

In [ ]:
# ===========================================
# Translation functions
# ===========================================

import re

SYSTEM_BLOCKS = [
    {
        "type": "text",
        "text": SYSTEM_PROMPT,
        "cache_control": {"type": "ephemeral"}
    },
]


def _is_tool_call(content: str) -> bool:
    """Detect if assistant message content is a function/tool call (not natural language).
    
    ToolACE uses patterns like: [func_name(param=value, ...)]
    """
    content = content.strip()
    if not content:
        return False
    # matches [function_name(params...)] or [func1(...), func2(...)]
    if re.match(r'^\[[\w.]+\(', content):
        return True
    return False


async def translate_text_async(text: str, semaphore: asyncio.Semaphore) -> str:
    """Translate a single text string to Indonesian."""
    async with semaphore:
        response = await async_client.messages.create(
            model=MODEL,
            system=SYSTEM_BLOCKS,
            messages=[{"role": "user", "content": text}],
            max_tokens=2048,
            temperature=0.3,
        )
        return response.content[0].text.strip()


async def translate_row_async(row: Dict, semaphore: asyncio.Semaphore) -> Dict:
    """Translate a single row — handles both single-turn and multi-turn formats.
    
    Single-turn (synthetic): inputs is plain text, translate directly.
    Multi-turn (Anthropic HH, ToolACE): inputs is a JSON array of messages,
    translate each message's content individually (skip tool calls).
    """
    translated_row = dict(row)
    translated_row["inputs_original"]   = row["inputs"]
    translated_row["language_original"] = row.get("language", "English")
    translated_row["language"]          = "Indonesian"
    
    if is_multiturn(row):
        messages = json.loads(row["inputs"])
        translated_messages = []
        
        for msg in messages:
            content = msg.get("content", "")
            
            # skip empty content or tool calls
            if not content.strip() or (msg["role"] == "assistant" and _is_tool_call(content)):
                translated_messages.append(msg)
                continue
            
            translated_content = await translate_text_async(content, semaphore)
            translated_messages.append({**msg, "content": translated_content})
        
        translated_row["inputs"] = json.dumps(translated_messages, ensure_ascii=False)
    else:
        translated_row["inputs"] = await translate_text_async(row["inputs"], semaphore)
    
    return translated_row


async def translate_and_save(rows: List[Dict], output_path: Path, dataset_name: str = ""):
    """Translate all rows with async concurrency, saving incrementally."""
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    completed = 0
    errors    = 0
    chunk_size = 50
    
    async def _safe_translate(idx: int, row: Dict) -> Dict:
        nonlocal completed, errors
        try:
            result = await translate_row_async(row, semaphore)
        except Exception as e:
            errors += 1
            print(f"  Error on row {idx}: {e}")
            result = dict(row)
            result["inputs_original"]   = row["inputs"]
            result["language_original"] = row.get("language", "English")
            result["inputs"]            = "[TRANSLATION_ERROR]"
            result["language"]          = "Indonesian"
        
        completed += 1
        if completed % 100 == 0:
            print(f"  [{dataset_name}] {completed}/{len(rows)} translated ({errors} errors)")
        return result
    
    for chunk_start in range(0, len(rows), chunk_size):
        chunk   = rows[chunk_start : chunk_start + chunk_size]
        tasks   = [_safe_translate(chunk_start + i, row) for i, row in enumerate(chunk)]
        results = await asyncio.gather(*tasks)
        
        with open(output_path, "a") as f:
            for r in results:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    
    print(f"  [{dataset_name}] Done: {completed} translated, {errors} errors -> {output_path.name}")


print("Translation functions ready (single-turn + multi-turn).")

---
## Part 3: Test on Small Sample

Translate 10 examples (5 high-stakes, 5 low-stakes) to check quality before scaling up.

In [9]:
# Sample 25 high-stakes + 25 low-stakes = 50 total for manual review
random.seed(42)
high_stakes = [r for r in raw_test if parse_label(r) == 1]
low_stakes  = [r for r in raw_test if parse_label(r) == 0]

sample = random.sample(high_stakes, 25) + random.sample(low_stakes, 25)
random.shuffle(sample)

print(f"Sample: {len(sample)} examples")
print(f"  High-stakes: {sum(1 for r in sample if parse_label(r) == 1)}")
print(f"  Low-stakes:  {sum(1 for r in sample if parse_label(r) == 0)}")

Sample: 50 examples
  High-stakes: 25
  Low-stakes:  25


In [ ]:
# Translate the 50-sample pilot using async for speed
import pandas as pd

async def translate_batch_async(rows: List[Dict], max_concurrent: int = MAX_CONCURRENT) -> List[Dict]:
    """Translate a batch of rows in memory (for pilot testing)."""
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def _one(idx, row):
        try:
            return await translate_row_async(row, semaphore)
        except Exception as e:
            print(f"  Error on row {idx}: {e}")
            result = dict(row)
            result["inputs_original"]   = row["inputs"]
            result["language_original"] = row.get("language", "English")
            result["inputs"]            = "[TRANSLATION_ERROR]"
            result["language"]          = "Indonesian"
            return result
    
    tasks   = [_one(i, row) for i, row in enumerate(rows)]
    results = await asyncio.gather(*tasks)
    n_errors = sum(1 for r in results if r.get("inputs") == "[TRANSLATION_ERROR]")
    print(f"  Completed {len(results)} translations ({n_errors} errors)")
    return list(results)


print(f"Translating {len(sample)} examples with {MAX_CONCURRENT} concurrent requests...")
t0 = time.time()
translated_sample = await translate_batch_async(sample, max_concurrent=MAX_CONCURRENT)
elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s ({elapsed/len(sample):.2f}s per example)")

# Check cache usage
test_response = client.messages.create(
    model=MODEL,
    system=SYSTEM_BLOCKS,
    messages=[{"role": "user", "content": "test"}],
    max_tokens=10,
)
print(f"\nCache check: {test_response.usage}")

In [ ]:
# Save as CSV for manual review
import pandas as pd

rows_for_csv = []
for i, row in enumerate(translated_sample):
    label = "HIGH" if parse_label(row) == 1 else "LOW"
    rows_for_csv.append({
        "idx":               i,
        "label":             label,
        "source_language":   row.get("language_original", "English"),
        "original":          row["inputs_original"],
        "indonesian":        row["inputs"],
        "acceptable":        "",  # for manual review
        "notes":             "",  # for manual notes
    })

df = pd.DataFrame(rows_for_csv)

csv_path = CACHE_DIR / "indonesian_pilot_50_review.csv"
df.to_csv(csv_path, index=False)

print(f"Saved {len(df)} translations to: {csv_path}")
print(f"\nPreview:")
print(df[["idx", "label", "source_language", "original", "indonesian"]].head(5).to_string(max_colwidth=80))

---
## Part 4: Evaluate Translation Quality

Look at the translations above and check:
1. Does the Indonesian sound natural (not translationese)?
2. Is the tone/severity preserved?
3. Are proper nouns and technical terms kept as-is?
4. Is anything added or lost?

If the quality is good, proceed to Part 5 to scale up.
If not, adjust `SYSTEM_PROMPT` or `FEW_SHOT_EXAMPLES` above and re-run.

In [12]:
# Quality evaluation function definition

import re

def evaluate_translation(original: str, id_text: str, source_lang: str, client: anthropic.Anthropic) -> Dict:
    """Ask Claude to evaluate a single translation on faithfulness and naturalness."""
    prompt = f"""Rate this {source_lang}-to-Indonesian translation on two scales (1-5):

Original ({source_lang}): {original}

Indonesian: {id_text}

Rate:
1. Faithfulness (1=wrong meaning, 5=proper meaning, context, and nuance preserved)
2. Naturalness (1=sounds robotic or overly word to word translation, 5=sounds like a native, professional translator)

Respond ONLY with a JSON object, no markdown fences: {{"faithfulness": N, "naturalness": N, "issues": "brief note or empty string"}}"""
    
    response = client.messages.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
        temperature=0,
    )
    
    raw = response.content[0].text.strip()
    
    # Strip markdown code fences if present
    raw = re.sub(r'^```(?:json)?\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)
    
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        return {"faithfulness": -1, "naturalness": -1, "issues": f"JSON parse failed: {e}. Raw: {raw[:200]}"}

In [13]:
# Run evaluation on translated sample
print("Evaluating translations...")
evals = []
for row in translated_sample:
    if row["inputs"] == "[TRANSLATION_ERROR]":
        continue
    source_lang = row.get("language_original", "English")
    ev = evaluate_translation(row["inputs_original"], row["inputs"], source_lang, client)
    evals.append(ev)
    time.sleep(0.1)

if evals:
    valid_faith = [e["faithfulness"] for e in evals if e["faithfulness"] > 0]
    valid_nat   = [e["naturalness"]  for e in evals if e["naturalness"]  > 0]
    if valid_faith:
        print(f"\nAvg faithfulness: {sum(valid_faith)/len(valid_faith):.1f}/5 ({len(valid_faith)}/{len(evals)} parsed)")
    if valid_nat:
        print(f"Avg naturalness:  {sum(valid_nat)/len(valid_nat):.1f}/5")
    
    for i, ev in enumerate(evals):
        if ev.get("issues"):
            print(f"  Example {i+1}: {ev['issues']}")

Evaluating translations...

Avg faithfulness: 4.2/5 (6/6 parsed)
Avg naturalness:  3.8/5
  Example 1: Translation is accurate but somewhat stiff. 'Ketenangan permukaan' is slightly awkward for 'comfortable constancy of surface'; 'penelitian kesehatan antariksa yang pelopor' is grammatically awkward (should be 'penelitian kesehatan antariksa pelopor' or 'penelitian pelopor kesehatan antariksa'); some phrases like 'merasakan kebersamaan' and 'merangkul kesempatan' sound overly literal rather than naturally Indonesian.
  Example 2: Minor: 'santai' is good but 'laid-back' also carries a sense of casual attitude toward responsibility. 'di internal' is slightly awkward; 'secara internal' would be more natural. Overall meaning and tone are well-preserved.
  Example 3: Minor: 'récupéré' (recovered/salvaged) translated as 'menyelamatkan' (save/rescue) is slightly stronger in tone; 'terganggu' (disrupted) is reasonable but 'dipahami salah' would be more literal. Overall meaning preserved well.
 

---
## Part 5: Translate All Datasets

Translate all three datasets to Indonesian:
1. **Synthetic** (2,000 examples) — 500 balanced sample (250 high + 250 low)
2. **Anthropic HH** (~2,984 examples) — full dataset, multi-turn
3. **ToolACE** (~734 examples) — full dataset, multi-turn with tool calls

Each dataset saves to a separate JSONL file. Supports resume if interrupted.

In [ ]:
# ===========================================
# Define translation jobs
# ===========================================

# Synthetic: balanced 500 sample (250 high + 250 low)
random.seed(42)
high_stakes = [r for r in raw_synthetic if parse_label(r) == 1]
low_stakes  = [r for r in raw_synthetic if parse_label(r) == 0]

N_PER_CLASS     = 250
synthetic_sample = random.sample(high_stakes, N_PER_CLASS) + random.sample(low_stakes, N_PER_CLASS)
random.shuffle(synthetic_sample)

# Anthropic HH and ToolACE: translate full datasets
OUTPUT_DIR = CACHE_DIR / "indonesian"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRANSLATION_JOBS = {
    "synthetic": {
        "data":        synthetic_sample,
        "output_path": OUTPUT_DIR / "synthetic_test_id.jsonl",
    },
    "anthropic": {
        "data":        raw_anthropic,
        "output_path": OUTPUT_DIR / "anthropic_test_id.jsonl",
    },
    "toolace": {
        "data":        raw_toolace,
        "output_path": OUTPUT_DIR / "toolace_test_id.jsonl",
    },
}

for name, job in TRANSLATION_JOBS.items():
    n_high = sum(1 for r in job["data"] if parse_label(r) == 1)
    n_low  = sum(1 for r in job["data"] if parse_label(r) == 0)
    print(f"{name:12s}: {len(job['data']):5d} to translate | {n_high:4d} high | {n_low:4d} low")
    print(f"{'':12s}  -> {job['output_path'].name}")

In [ ]:
# ===========================================
# Run translations with resume support
# ===========================================

t0_total = time.time()

for name, job in TRANSLATION_JOBS.items():
    output_path = job["output_path"]
    all_rows    = job["data"]
    
    # Check for existing partial results to resume
    already_done = set()
    if output_path.exists():
        with open(output_path) as f:
            for line in f:
                row = json.loads(line)
                already_done.add(row.get("ids", ""))
        print(f"[{name}] Resuming: {len(already_done)} already translated")
    
    remaining = [r for r in all_rows if r.get("ids", "") not in already_done]
    
    if not remaining:
        print(f"[{name}] All {len(all_rows)} already done, skipping.")
        continue
    
    print(f"[{name}] Translating {len(remaining)} remaining ({len(already_done)} done)...")
    t0 = time.time()
    await translate_and_save(remaining, output_path, dataset_name=name)
    elapsed = time.time() - t0
    print(f"[{name}] Elapsed: {elapsed:.1f}s ({elapsed/max(len(remaining),1):.2f}s per row)\n")

total_elapsed = time.time() - t0_total
print(f"All translations complete in {total_elapsed:.0f}s ({total_elapsed/60:.1f} min)")

---
## Part 6: Verify Outputs

Check all translated datasets for completeness and errors.

In [ ]:
# Verify all outputs
print("=" * 70)
print("TRANSLATION SUMMARY")
print("=" * 70)

for name, job in TRANSLATION_JOBS.items():
    output_path = job["output_path"]
    expected    = len(job["data"])
    
    if not output_path.exists():
        print(f"\n{name}: FILE NOT FOUND ({output_path})")
        continue
    
    translated = load_jsonl(output_path)
    n_errors   = sum(1 for r in translated if r.get("inputs") == "[TRANSLATION_ERROR]")
    n_high     = sum(1 for r in translated if parse_label(r) == 1)
    n_low      = sum(1 for r in translated if parse_label(r) == 0)
    
    print(f"\n{name}:")
    print(f"  Expected:    {expected}")
    print(f"  Translated:  {len(translated)}")
    print(f"  High-stakes: {n_high}")
    print(f"  Low-stakes:  {n_low}")
    print(f"  Errors:      {n_errors}")
    print(f"  File:        {output_path}")
    
    if n_errors > 0:
        print(f"  WARNING: {n_errors} translation errors need re-running")
    
    # Show first example
    if translated:
        row = translated[0]
        label = "HIGH" if parse_label(row) == 1 else "LOW"
        if is_multiturn(row):
            msgs = json.loads(row["inputs"])
            user_msg = next((m["content"] for m in msgs if m["role"] == "user"), "")
            print(f"  Sample [{label}]: {user_msg[:120]}...")
        else:
            print(f"  Sample [{label}]: {row['inputs'][:120]}...")

print("\n" + "=" * 70)
print("Output directory:", OUTPUT_DIR)
print("=" * 70)

---
## Part 7: Download / Transfer (Colab only)

If running on Colab, download the translated files for local use.

In [ ]:
if ENV == "colab":
    from google.colab import files
    for name, job in TRANSLATION_JOBS.items():
        if job["output_path"].exists():
            files.download(str(job["output_path"]))
            print(f"Downloaded: {job['output_path'].name}")
else:
    print("Files ready locally:")
    for name, job in TRANSLATION_JOBS.items():
        exists = job["output_path"].exists()
        status = "OK" if exists else "MISSING"
        print(f"  [{status}] {job['output_path']}")